# Olympic Athlete Analysis — Exploratory Data Analysis (EDA)

### Group Project | Python • Pandas • Matplotlib • Seaborn

**Dataset:** Olympic athlete-event records from Athens 1896 to Rio 2016  
**Objective:** Clean the dataset, explore participation and athlete characteristics, analyze medal performance, identify patterns, and communicate evidence-based insights.

> **Important interpretation note:** Each row represents an **athlete-event record**, not a unique person. One athlete can therefore appear in many rows. For participation questions, this notebook reports both record counts and unique-athlete counts where useful.

## 1. Project Objectives

This project follows the provided EDA guidelines:

1. Understand the dataset and its variables.
2. Identify missing values, duplicate records, and potential outliers.
3. Clean the data using transparent rules.
4. Explore participation, gender, season, sport, NOC, medals, age, height, and weight.
5. Investigate relationships such as age vs. weight.
6. Summarize findings with charts and concise interpretations.

## 2. Business Questions

1. How has Olympic participation changed over time?
2. What is the distribution of male and female participation?
3. Which sports have the highest athlete participation?
4. Which NOCs have the highest Olympic participation?
5. Which NOCs have the most medal records?
6. What is the typical age of Olympic athletes?
7. How are athlete height and weight distributed?
8. How does medal performance differ by gender?
9. How does participation differ between Summer and Winter Olympics?
10. Which sports have the highest number of medal records?
11. How has female participation changed over time?
12. Is there a visible relationship between athlete age and weight?

## 3. Import Libraries

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 4. Load the Dataset

The notebook first looks for `athlete_data.csv` in the project's `data` folder, then checks the current folder as a fallback.

In [ ]:
possible_paths = [
    os.path.join("data", "athlete_data.csv"),
    "athlete_data.csv",
    os.path.join("data", "athlete_data_demo.csv")
]

DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "No Olympic CSV found. Place the permitted original dataset at data/athlete_data.csv, "
        "or run the included synthetic demonstration file data/athlete_data_demo.csv."
    )

IS_SYNTHETIC_DEMO = DATA_PATH.endswith("athlete_data_demo.csv")
if IS_SYNTHETIC_DEMO:
    print("DEMO MODE: synthetic sample data only. These results are not Olympic history findings.")

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")


## 5. Dataset Understanding

In [ ]:
display(df.head())
display(df.tail())
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nDataset information:")
df.info()

In [ ]:
summary = pd.DataFrame({
    "Rows": [len(df)],
    "Columns": [df.shape[1]],
    "Unique athletes (ID)": [df["ID"].nunique()],
    "Unique sports": [df["Sport"].nunique()],
    "Unique NOCs": [df["NOC"].nunique()],
    "Unique events": [df["Event"].nunique()],
    "Olympic Games": [df["Games"].nunique()],
    "Year range": [f"{df['Year'].min()}–{df['Year'].max()}"]
})
display(summary)

### Dataset interpretation

The supplied dataset contains 271,116 rows and 15 columns. The source documentation states that each row is an individual athlete competing in an individual Olympic event. Therefore, row counts measure **athlete-event records**, while `ID.nunique()` measures unique athletes.

## 6. Data Quality Assessment

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2)
}).sort_values("Missing %", ascending=False)
display(missing)

In [ ]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_count:,}")

print("\nSex distribution:")
display(df["Sex"].value_counts())
print("\nSeason distribution:")
display(df["Season"].value_counts())
print("\nMedal distribution including missing values:")
display(df["Medal"].value_counts(dropna=False))

## 7. Data Cleaning and Feature Engineering

### Cleaning rules

- Remove exact duplicate rows.
- Treat missing `Medal` as **No Medal** because a blank medal means the athlete-event record did not receive Gold, Silver, or Bronze.
- Impute missing `Age` using the median of the same `Sport` + `Sex` group.
- Impute missing `Height` and `Weight` using the median of the same `Sport` + `Sex` group; remaining gaps use the overall median.
- Investigate potential outliers with boxplots. Do **not** automatically delete them because extreme values may represent valid Olympic records.
- Create `Age_Group` and `Won_Medal` for analysis.

These rules are based on the supplied notebook and project guidelines.

In [ ]:
df_clean = df.copy()

# 1. Remove exact duplicate rows
df_clean = df_clean.drop_duplicates().copy()

# 2. Medal: missing means no medal
df_clean["Medal"] = df_clean["Medal"].fillna("No Medal")

# 3. Age: Sport + Sex median, then overall median if needed
df_clean["Age"] = df_clean["Age"].fillna(
    df_clean.groupby(["Sport", "Sex"])["Age"].transform("median")
)
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())

# 4. Height: Sport + Sex median, then overall median
df_clean["Height"] = df_clean["Height"].fillna(
    df_clean.groupby(["Sport", "Sex"])["Height"].transform("median")
)
df_clean["Height"] = df_clean["Height"].fillna(df_clean["Height"].median())

# 5. Weight: Sport + Sex median, then overall median
df_clean["Weight"] = df_clean["Weight"].fillna(
    df_clean.groupby(["Sport", "Sex"])["Weight"].transform("median")
)
df_clean["Weight"] = df_clean["Weight"].fillna(df_clean["Weight"].median())

# 6. Age groups
bins = [0, 15, 20, 25, 30, 35, 40, 50, np.inf]
labels = ["Under 16", "16–20", "21–25", "26–30", "31–35", "36–40", "41–50", "51+"]
df_clean["Age_Group"] = pd.cut(df_clean["Age"], bins=bins, labels=labels, right=True)

# 7. Medal flag
df_clean["Won_Medal"] = df_clean["Medal"].ne("No Medal")

print(f"Original rows: {len(df):,}")
print(f"Cleaned rows:  {len(df_clean):,}")
print(f"Rows removed:  {len(df)-len(df_clean):,}")
print(f"Final columns: {len(df_clean.columns)}")
print(f"Remaining missing values: {int(df_clean.isna().sum().sum())}")

In [ ]:
cleaning_summary = pd.DataFrame({
    "Metric": ["Original rows", "Duplicate rows removed", "Final rows", "Original columns", "Final columns", "Remaining missing cells"],
    "Value": [len(df), duplicate_count, len(df_clean), df.shape[1], df_clean.shape[1], int(df_clean.isna().sum().sum())]
})
display(cleaning_summary)

## 8. Question 1 — Olympic Participation Over Time

Because one row is an athlete-event record, the first chart uses record counts. A second line shows unique athletes to reduce the effect of athletes competing in multiple events.

In [ ]:
year_records = df_clean.groupby("Year").size()
year_athletes = df_clean.groupby("Year")["ID"].nunique()

fig, ax = plt.subplots(figsize=(12, 5))
year_records.plot(ax=ax, marker="o", linewidth=1.5, label="Athlete-event records")
year_athletes.plot(ax=ax, marker=".", linewidth=1.5, label="Unique athletes")
ax.set_title("Olympic Participation Over Time")
ax.set_xlabel("Year")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

print("Highest record-count year:", int(year_records.idxmax()), f"({year_records.max():,} records)")
print("Highest unique-athlete year:", int(year_athletes.idxmax()), f"({year_athletes.max():,} athletes)")

**Insight:** Participation expanded substantially across the historical period, although the two measures are not identical because athletes can compete in multiple events.

## 9. Question 2 — Male and Female Participation

In [ ]:
sex_records = df_clean["Sex"].value_counts()
sex_pct = (sex_records / sex_records.sum() * 100).round(2)
display(pd.DataFrame({"Records": sex_records, "Percentage": sex_pct}))

plt.figure(figsize=(7, 5))
sex_records.sort_values().plot(kind="bar")
plt.title("Athlete-Event Records by Sex")
plt.xlabel("Sex")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

## 10. Question 3 — Sports with the Highest Participation

In [ ]:
top_sports = df_clean["Sport"].value_counts().head(10).sort_values()
display(top_sports.sort_values(ascending=False).to_frame("Athlete-event records"))

plt.figure(figsize=(10, 6))
top_sports.plot(kind="barh")
plt.title("Top 10 Sports by Athlete-Event Records")
plt.xlabel("Records")
plt.ylabel("Sport")
plt.tight_layout()
plt.show()

## 11. Question 4 — NOCs with the Highest Participation

In [ ]:
top_noc = df_clean["NOC"].value_counts().head(10).sort_values()
display(top_noc.sort_values(ascending=False).to_frame("Athlete-event records"))

plt.figure(figsize=(10, 6))
top_noc.plot(kind="barh")
plt.title("Top 10 NOCs by Athlete-Event Records")
plt.xlabel("Records")
plt.ylabel("NOC")
plt.tight_layout()
plt.show()

**Note:** The supplied dataset contains three-letter NOC codes. This notebook reports the codes directly rather than assigning country names that are not included in the supplied columns.

## 12. Question 5 — NOCs with the Most Medal Records

In [ ]:
medal_records = df_clean[df_clean["Won_Medal"]]
top_medal_noc = medal_records["NOC"].value_counts().head(10).sort_values()
display(top_medal_noc.sort_values(ascending=False).to_frame("Medal records"))

plt.figure(figsize=(10, 6))
top_medal_noc.plot(kind="barh")
plt.title("Top 10 NOCs by Medal Records")
plt.xlabel("Medal records")
plt.ylabel("NOC")
plt.tight_layout()
plt.show()

## 13. Question 6 — Typical Age of Olympic Athletes

In [ ]:
age_stats = df_clean["Age"].describe()
display(age_stats.to_frame("Age"))

plt.figure(figsize=(10, 5))
plt.hist(df_clean["Age"], bins=30, edgecolor="black", alpha=0.75)
plt.title("Age Distribution of Olympic Athletes")
plt.xlabel("Age")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 3))
plt.boxplot(df_clean["Age"], vert=False)
plt.title("Age Boxplot")
plt.xlabel("Age")
plt.tight_layout()
plt.show()

**Insight:** The cleaned age distribution is centered in the mid-20s. The observed range includes ages from 10 to 97. Extreme observations were retained rather than removed because the source project treats them as potentially valid Olympic records.

## 14. Question 7 — Height and Weight Distributions

In [ ]:
profile_stats = df_clean[["Height", "Weight"]].describe().T
profile_stats["IQR"] = profile_stats["75%"] - profile_stats["25%"]
display(profile_stats)

plt.figure(figsize=(9, 5))
plt.hist(df_clean["Height"], bins=30, edgecolor="black", alpha=0.75)
plt.title("Height Distribution")
plt.xlabel("Height (cm)")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
plt.hist(df_clean["Weight"], bins=30, edgecolor="black", alpha=0.75)
plt.title("Weight Distribution")
plt.xlabel("Weight (kg)")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(df_clean["Height"], vert=False)
axes[0].set_title("Height Boxplot")
axes[0].set_xlabel("Height (cm)")
axes[1].boxplot(df_clean["Weight"], vert=False)
axes[1].set_title("Weight Boxplot")
axes[1].set_xlabel("Weight (kg)")
plt.tight_layout()
plt.show()

## 15. Question 8 — Medal Performance by Gender

A simple record count is not enough to compare medal outcomes because male and female record totals differ. Therefore, the notebook shows both medal counts and the percentage of records that won a medal within each sex.

In [ ]:
gender_medal_counts = pd.crosstab(df_clean["Sex"], df_clean["Medal"])
display(gender_medal_counts)

gender_medal_rate = df_clean.groupby("Sex")["Won_Medal"].mean().mul(100).round(2)
display(gender_medal_rate.to_frame("Medal-winning record %"))

plot_data = gender_medal_counts.reindex(columns=["Gold", "Silver", "Bronze", "No Medal"])
plot_data.plot(kind="bar", stacked=True, figsize=(10, 5))
plt.title("Medal Outcomes by Sex")
plt.xlabel("Sex")
plt.ylabel("Records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Interpretation:** Male records are more numerous in the dataset, so their absolute medal counts are also higher. The within-group medal-winning percentage provides a second perspective and should be interpreted as a record-level rate, not an athlete-level probability.

## 16. Question 9 — Summer vs Winter Participation

In [ ]:
season_records = df_clean["Season"].value_counts()
season_athletes = df_clean.groupby("Season")["ID"].nunique()
season_summary = pd.DataFrame({"Records": season_records, "Unique athletes": season_athletes})
display(season_summary)

plt.figure(figsize=(7, 5))
season_records.sort_values().plot(kind="bar")
plt.title("Athlete-Event Records by Olympic Season")
plt.xlabel("Season")
plt.ylabel("Records")
plt.tight_layout()
plt.show()

**Important historical context:** The supplied dataset documentation notes that Summer and Winter Games were held in the same year up to 1992, after which the Games were staggered beginning with Winter 1994 and Summer 1996. Therefore, year-by-year season comparisons must account for this schedule change.

## 17. Question 10 — Sports with the Most Medal Records

In [ ]:
top_medal_sports = medal_records["Sport"].value_counts().head(10).sort_values()
display(top_medal_sports.sort_values(ascending=False).to_frame("Medal records"))

plt.figure(figsize=(10, 6))
top_medal_sports.plot(kind="barh")
plt.title("Top 10 Sports by Medal Records")
plt.xlabel("Medal records")
plt.ylabel("Sport")
plt.tight_layout()
plt.show()

## 18. Question 11 — Female Participation Over Time

In [ ]:
female_records = df_clean.loc[df_clean["Sex"] == "F"].groupby("Year").size()
female_athletes = df_clean.loc[df_clean["Sex"] == "F"].groupby("Year")["ID"].nunique()

fig, ax = plt.subplots(figsize=(12, 5))
female_records.plot(ax=ax, marker="o", label="Female athlete-event records")
female_athletes.plot(ax=ax, marker=".", label="Unique female athletes")
ax.set_title("Female Olympic Participation Over Time")
ax.set_xlabel("Year")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

print("Female records in 1896:", int(female_records.get(1896, 0)))
print("Female records in 2016:", int(female_records.get(2016, 0)))

**Insight:** Female participation increases markedly over the historical period represented in the dataset. The comparison is descriptive and reflects athlete-event records as well as unique-athlete counts.

## 19. Question 12 — Age and Weight Relationship

In [ ]:
corr = df_clean[["Age", "Weight"]].corr().loc["Age", "Weight"]
print(f"Pearson correlation between Age and Weight: {corr:.3f}")

sample = df_clean[["Age", "Weight"]].sample(min(20000, len(df_clean)), random_state=42)
plt.figure(figsize=(10, 6))
plt.scatter(sample["Age"], sample["Weight"], alpha=0.25, s=20)
plt.title("Age vs Weight (20,000-record sample where available)")
plt.xlabel("Age")
plt.ylabel("Weight (kg)")
plt.tight_layout()
plt.show()

**Interpretation:** Correlation measures linear association only; it does not establish causation. Athlete weight also varies substantially by sport, sex, and event, so any age-weight relationship should not be interpreted in isolation.

## 20. Medal Distribution Summary

In [ ]:
medal_summary = df_clean["Medal"].value_counts()
medal_pct = df_clean["Medal"].value_counts(normalize=True).mul(100).round(2)
medal_table = pd.DataFrame({"Records": medal_summary, "Percentage": medal_pct})
display(medal_table)

plt.figure(figsize=(7, 7))
plt.pie(medal_summary, labels=medal_summary.index, autopct="%1.1f%%", startangle=90)
plt.title("Medal Distribution")
plt.tight_layout()
plt.show()

## 21. Key Findings

- The dataset covers Olympic athlete-event records from **1896 to 2016** and contains **66 sports**.
- The original dataset has **271,116 rows** and **15 columns**.
- **1,385 exact duplicate rows** are removed during cleaning, leaving **269,731 records**.
- Missing medal values are treated as **No Medal**; age, height, and weight are imputed using sport-and-sex medians followed by overall medians where necessary.
- Male athlete-event records outnumber female records in the full historical dataset, while female participation grows over time.
- Athletics, Gymnastics, and Swimming are among the sports with the highest numbers of athlete-event records.
- USA, FRA, and GBR are among the NOCs with the highest participation record counts in the supplied data.
- USA, URS, and GER are among the NOCs with the highest medal-record counts in the supplied data.
- The age distribution is centered in the mid-20s, with a broad observed range.
- Height and weight contain substantial missingness before cleaning, making explicit imputation important for complete exploratory analysis.
- Summer records substantially outnumber Winter records in the full dataset.
- Medal counts and participation counts are record-level measures; they should not automatically be interpreted as unique-athlete performance measures.

## 22. Limitations

1. The dataset is historical and ends in 2016; it does not represent Olympics after Rio 2016.
2. Rows are athlete-event records, so repeated appearances by the same athlete can affect counts.
3. NOC is a three-letter code; country names are not directly included in the supplied columns.
4. Median imputation is useful for EDA completeness but may reduce natural variation in the imputed values.
5. Outliers were retained because they may be valid observations; this means extreme values can influence summary statistics.
6. Correlation between age and weight does not imply causation and can be confounded by sport and sex.

## 23. Conclusion

This EDA converts a large Olympic athlete-event dataset into a structured set of descriptive findings. The workflow demonstrates the full EDA process: understanding the data, measuring data-quality issues, cleaning and feature engineering, visualizing distributions and trends, comparing groups, and documenting limitations.

The cleaned dataset is suitable for deeper analysis such as sport-specific medal rates, NOC trends, gender participation patterns, event-level comparisons, and predictive modeling.

In [ ]:
# Optional: save the cleaned dataset for downstream analysis
auto_output = "olympic_athlete_cleaned.csv"
df_clean.to_csv(auto_output, index=False)
print(f"Saved cleaned dataset: {auto_output}")